# Experiment 005 — Compositional Cell Graph

Use the existing Kaggle notebook [bingjiangli/minicells](https://www.kaggle.com/code/bingjiangli/minicells); do not create another notebook. Copy these cells into that notebook. This is a thin manual entrypoint: model, training, accounting and analysis live in the project-owned Python package.

The proposal is currently **planned and unregistered**. The default mode runs CPU preflight only. Full training is refused until the holdout, probe rule, seed matrix, matching rules and measurement gates are frozen in the protocol and config. Training has no application wall-clock cap.

Training mode uses the Kaggle notebook's existing `HF_TOKEN` and `GITHUB_TOKEN` secrets, checks GitHub write access before asset preparation/training, and publishes immutable arm evidence through the repository's existing Git askpass publisher. It does not call Kaggle notebook or dataset upload APIs.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import re
import subprocess
import sys
import threading

# Keep preflight as the default. Training requires a frozen full source SHA.
MODE = 'preflight'  # preflight | dry-run | train | train_pair | analyze | push_existing
SOURCE_COMMIT = '3a073bf5e56941a42ce40af85a7c405cb9d37c18'  # replace with the audit-fix / registered commit before running
SEED = 42
ARM = 'G0'
ARM_PAIR = ('M0', 'P0')  # train these two arms concurrently, one per T4
DEVICE = 'cuda:0'
KAGGLE_NOTEBOOK_URL = 'https://www.kaggle.com/code/bingjiangli/minicells'
REPOSITORY_URL = 'https://github.com/ArcheLabs/mini-cells.git'
ROOT = Path('/kaggle/working/minicells-exp005')
RUN_ROOT = Path('/kaggle/working/minicells-exp005-runs')
ASSET_ROOT = Path('/kaggle/working/minicells-exp005-assets')

if MODE not in {'preflight', 'dry-run', 'train', 'train_pair', 'analyze', 'push_existing'}:
    raise ValueError('Choose a supported preflight, training, analysis, or publish-retry mode')
if MODE in {'train', 'train_pair', 'analyze', 'push_existing'} and not re.fullmatch(r'[0-9a-f]{40}', SOURCE_COMMIT):
    raise ValueError('Training/analysis requires a pinned full lowercase SOURCE_COMMIT SHA')

def git(*args):
    return subprocess.run(['git', *args], cwd=ROOT, check=True, text=True,
                          stdout=subprocess.PIPE).stdout.strip()

if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth=1', '--filter=blob:none',
                    '--branch', 'codex/research-project-first-layout', '--single-branch', REPOSITORY_URL, str(ROOT)],
                   check=True)
elif not (ROOT / '.git').exists():
    raise RuntimeError(f'Refusing to overwrite non-repository path: {ROOT}')
elif git('status', '--porcelain'):
    raise RuntimeError('The existing checkout is dirty; preserve its files before continuing.')

if SOURCE_COMMIT == 'main':
    subprocess.run(['git', 'fetch', '--depth=1', 'origin', 'main'], cwd=ROOT, check=True)
    SOURCE_SHA = git('rev-parse', 'FETCH_HEAD')
else:
    subprocess.run(['git', 'fetch', '--depth=1', 'origin', SOURCE_COMMIT], cwd=ROOT, check=True)
    SOURCE_SHA = SOURCE_COMMIT
RESULT_BRANCH_NAME = f'results/005-compositional-cell-graph-{SOURCE_SHA[:12]}'
CURRENT_BRANCH = git('rev-parse', '--abbrev-ref', 'HEAD')
LOCAL_RESULT_BRANCH = subprocess.run(['git', 'show-ref', '--verify', '--quiet',
                                    f'refs/heads/{RESULT_BRANCH_NAME}'], cwd=ROOT, check=False).returncode == 0
if LOCAL_RESULT_BRANCH:
    subprocess.run(['git', 'checkout', RESULT_BRANCH_NAME], cwd=ROOT, check=True)
elif CURRENT_BRANCH != RESULT_BRANCH_NAME:
    subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=ROOT, check=True)
os.environ['PYTHONPATH'] = os.pathsep.join((str(ROOT / 'src'), str(ROOT)))
RUNNER = ROOT / 'scripts/research/run_experiment_005.py'
PUBLISHER = ROOT / 'scripts/research/publish_experiment_005.py'
CONFIG_PATH = ROOT / 'research/08-hybrid-clm/experiments/005-compositional-cell-graph/config/proposed.json'
PROTOCOL_PATH = ROOT / 'research/08-hybrid-clm/experiments/005-compositional-cell-graph/protocol.json'
CONFIG = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
PROTOCOL = json.loads(PROTOCOL_PATH.read_text(encoding='utf-8'))

if MODE in {'preflight', 'dry-run'}:
    subprocess.run([sys.executable, str(RUNNER), '--mode', MODE, '--config', str(CONFIG_PATH)],
                   cwd=ROOT, check=True)
elif MODE == 'push_existing':
    from kaggle_secrets import UserSecretsClient
    try:
        os.environ['GITHUB_TOKEN'] = UserSecretsClient().get_secret('GITHUB_TOKEN').strip()
    except Exception as exc:
        raise RuntimeError('Cannot read Kaggle Secret GITHUB_TOKEN in the original notebook.') from exc
    if not os.environ['GITHUB_TOKEN']:
        raise RuntimeError('Kaggle Secret GITHUB_TOKEN is empty.')
    subprocess.run([sys.executable, str(PUBLISHER), '--source-commit', SOURCE_SHA,
                    '--secret-name', 'GITHUB_TOKEN', '--push-existing'], cwd=ROOT, check=True)
    os.environ.pop('GITHUB_TOKEN', None)
elif MODE == 'analyze':
    from kaggle_secrets import UserSecretsClient
    try:
        os.environ['GITHUB_TOKEN'] = UserSecretsClient().get_secret('GITHUB_TOKEN').strip()
    except Exception as exc:
        raise RuntimeError('Cannot read Kaggle Secret GITHUB_TOKEN in the original notebook.') from exc
    if not os.environ['GITHUB_TOKEN']:
        raise RuntimeError('Kaggle Secret GITHUB_TOKEN is empty.')
    subprocess.run([sys.executable, str(PUBLISHER), '--source-commit', SOURCE_SHA,
                    '--secret-name', 'GITHUB_TOKEN', '--preflight-only'], cwd=ROOT, check=True)
    ARCHIVE = ROOT / 'research/08-hybrid-clm/experiments/005-compositional-cell-graph/artifacts' / SOURCE_SHA
    ANALYSIS_DIR = Path('/kaggle/working/minicells-exp005-analysis') / SOURCE_SHA
    analysis_process = subprocess.run([sys.executable, str(ROOT / 'scripts/research/analyze_experiment_005.py'),
                                      '--archive', str(ARCHIVE), '--output-dir', str(ANALYSIS_DIR)],
                                     cwd=ROOT, check=True, text=True, stdout=subprocess.PIPE)
    ANALYSIS_OUTPUT = Path(json.loads(analysis_process.stdout)['analysis_report'])
    subprocess.run([sys.executable, str(PUBLISHER), '--source-commit', SOURCE_SHA,
                    '--secret-name', 'GITHUB_TOKEN', '--analysis-report',
                    str(ANALYSIS_OUTPUT)], cwd=ROOT, check=True)
    os.environ.pop('GITHUB_TOKEN', None)
    print(json.loads(ANALYSIS_OUTPUT.read_text(encoding='utf-8')))
elif MODE in {'train', 'train_pair'}:
    if PROTOCOL.get('status') != 'REGISTERED' or CONFIG.get('protocol_status') != 'registered':
        raise RuntimeError('Experiment 005 is not registered; run preflight/dry-run only.')
    selected_arms = (ARM,) if MODE == 'train' else tuple(ARM_PAIR)
    if MODE == 'train_pair' and len(selected_arms) != 2:
        raise ValueError('train_pair requires exactly two arms')
    if len(set(selected_arms)) != len(selected_arms) or any(arm not in CONFIG['arms'] for arm in selected_arms):
        raise ValueError('Select distinct arms from the registered matrix')
    development = set(CONFIG['seeds']['development'])
    confirmation = set(CONFIG['seeds']['confirmation'])
    if development & confirmation or SEED not in development | confirmation:
        raise ValueError('SEED is not in the disjoint registered development/confirmation matrix')
    if MODE == 'train_pair':
        import torch
        if torch.cuda.device_count() < 2:
            raise RuntimeError('train_pair requires two visible GPUs; no training started.')
    from kaggle_secrets import UserSecretsClient
    secret_client = UserSecretsClient()
    for secret_name in ('GITHUB_TOKEN', 'HF_TOKEN'):
        try:
            secret_value = secret_client.get_secret(secret_name).strip()
        except Exception as exc:
            raise RuntimeError(f'Cannot read Kaggle Secret {secret_name} in the original notebook.') from exc
        if not secret_value:
            raise RuntimeError(f'Kaggle Secret {secret_name} is empty.')
        os.environ[secret_name] = secret_value
        del secret_value
    # Check write access before spending time on asset preparation or GPU work.
    subprocess.run([sys.executable, str(PUBLISHER), '--source-commit', SOURCE_SHA,
                    '--secret-name', 'GITHUB_TOKEN', '--preflight-only'], cwd=ROOT, check=True)
    for module, package in (('datasets', 'datasets>=2.18,<5'),
                            ('tokenizers', 'tokenizers>=0.15,<1')):
        if importlib.util.find_spec(module) is None:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                            '--disable-pip-version-check', package], check=True)
    expected_assets_sha = CONFIG['data'].get('assets_manifest_sha256')
    expected_probe_count = CONFIG['training'].get('edge_refresh_probe_count')
    probe_seed = CONFIG['training'].get('probe_seed')
    if not expected_assets_sha or expected_probe_count is None or probe_seed is None:
        raise RuntimeError('Asset/probe identities are not frozen in the registered config.')
    ASSETS = ASSET_ROOT / expected_assets_sha
    ASSET_MANIFEST = ASSETS / 'assets-manifest.json'
    if not ASSET_MANIFEST.is_file():
        subprocess.run([sys.executable, str(RUNNER), '--mode', 'prepare-assets',
                        '--config', str(CONFIG_PATH), '--output', str(ASSETS),
                        '--probe-window-count', str(expected_probe_count),
                        '--probe-seed', str(probe_seed)], cwd=ROOT, check=True)
    actual_assets_sha = hashlib.sha256(ASSET_MANIFEST.read_bytes()).hexdigest()
    if actual_assets_sha != expected_assets_sha:
        raise RuntimeError('Prepared assets differ from the registered manifest; no training started.')
    os.environ.pop('HF_TOKEN', None)
    run_outputs = {arm: RUN_ROOT / SOURCE_SHA / f'seed-{SEED}' / arm for arm in selected_arms}
    completed_arms = {arm for arm, path in run_outputs.items() if (path / 'metrics.json').is_file()}
    pending_arms = [arm for arm in selected_arms if arm not in completed_arms]
    successful_arms = set(completed_arms)
    failed_arms = {}
    if MODE == 'train':
        arm = selected_arms[0]
        if arm not in completed_arms:
            subprocess.run([sys.executable, str(RUNNER), '--mode', 'train', '--config',
                            str(CONFIG_PATH), '--arm', arm, '--seed', str(SEED),
                            '--source-commit', SOURCE_SHA, '--assets', str(ASSETS),
                            '--asset-manifest-sha256', expected_assets_sha,
                            '--output', str(run_outputs[arm]), '--device', DEVICE], cwd=ROOT, check=True)
            successful_arms.add(arm)
        else:
            print(f'{arm} training is already complete; resuming profile/publication only.')
    elif pending_arms:
        process_rows, log_threads = [], []
        def stream_arm_output(arm_name, process, log_path):
            with log_path.open('w', encoding='utf-8') as log_handle:
                for line in process.stdout:
                    log_handle.write(line)
                    log_handle.flush()
                    print(f'[{arm_name}] {line}', end='', flush=True)
        for gpu_index, arm in enumerate(pending_arms):
            run_output = run_outputs[arm]
            log_path = RUN_ROOT / SOURCE_SHA / f'seed-{SEED}-{arm}.log'
            log_path.parent.mkdir(parents=True, exist_ok=True)
            command = [sys.executable, str(RUNNER), '--mode', 'train', '--config',
                       str(CONFIG_PATH), '--arm', arm, '--seed', str(SEED),
                       '--source-commit', SOURCE_SHA, '--assets', str(ASSETS),
                       '--asset-manifest-sha256', expected_assets_sha,
                       '--output', str(run_output), '--device', f'cuda:{gpu_index}']
            process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True, bufsize=1)
            process_rows.append((arm, process))
            thread = threading.Thread(target=stream_arm_output,
                                     args=(arm, process, log_path), daemon=True)
            thread.start()
            log_threads.append(thread)
        process_status = {arm: process.wait() for arm, process in process_rows}
        for thread in log_threads:
            thread.join()
        for arm, status in process_status.items():
            if status == 0:
                successful_arms.add(arm)
            else:
                failed_arms[arm] = status
    # Decode timings are collected serially after every concurrent training process has exited.
    for arm in selected_arms:
        if arm not in successful_arms:
            continue
        run_output = run_outputs[arm]
        if not (run_output / 'decode-profile.json').is_file():
            subprocess.run([sys.executable, str(RUNNER), '--mode', 'profile', '--config',
                            str(CONFIG_PATH), '--arm', arm, '--seed', str(SEED),
                            '--source-commit', SOURCE_SHA, '--assets', str(ASSETS),
                            '--asset-manifest-sha256', expected_assets_sha,
                            '--output', str(run_output), '--device', 'cuda:0'], cwd=ROOT, check=True)
        subprocess.run([sys.executable, str(PUBLISHER), '--source-commit', SOURCE_SHA,
                        '--secret-name', 'GITHUB_TOKEN', '--seed', str(SEED), '--arm', arm,
                        '--output', str(run_output), '--assets-manifest', str(ASSET_MANIFEST),
                        '--kaggle-notebook-url', KAGGLE_NOTEBOOK_URL], cwd=ROOT, check=True)
    os.environ.pop('GITHUB_TOKEN', None)
    if failed_arms:
        raise RuntimeError(f'Completed arms were preserved/published; resume only failed arms: {failed_arms}')
    print({'status': 'ARM_EVIDENCE_PUBLISHED', 'source_commit': SOURCE_SHA,
           'seed': SEED, 'arms': sorted(successful_arms)})